<a href="https://colab.research.google.com/github/Anudeepkonka/airline-ticket-automation/blob/main/Airline_Ticket_Automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install spacy scikit-learn pandas nltk streamlit pyngrok joblib
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 105.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 92.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:
import pandas as pd
import numpy as np
import random

# Sample cities and names
cities = ["Hyderabad", "Delhi", "Mumbai", "Chennai", "Bangalore", "Kolkata"]
names = ["John", "Rahul", "Priya", "Anita", "Arjun", "Sneha"]

emails = []
intents = []

# Book Flight
for _ in range(40):
    emails.append(
        f"Book a flight for {random.choice(names)} from {random.choice(cities)} to {random.choice(cities)} tomorrow."
    )
    intents.append("Book Flight")

# Cancel Flight
for _ in range(40):
    emails.append(
        f"Please cancel my booking ABC{random.randint(100,999)}."
    )
    intents.append("Cancel Flight")

# Reschedule Flight
for _ in range(40):
    emails.append(
        f"Reschedule my flight to next Monday."
    )
    intents.append("Reschedule Flight")

# General Inquiry
for _ in range(40):
    emails.append(
        "What is the baggage allowance?"
    )
    intents.append("General Inquiry")

# Create DataFrame
df = pd.DataFrame({
    "email": emails,
    "intent": intents
})

# Shuffle data
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save dataset
df.to_csv("airline_emails.csv", index=False)

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (160, 2)


,email,intent
0,Reschedule my flight to next Monday.,Reschedule Flight
1,Reschedule my flight to next Monday.,Reschedule Flight
2,What is the baggage allowance?,General Inquiry
3,Please cancel my booking ABC689.,Cancel Flight
4,Reschedule my flight to next Monday.,Reschedule Flight


In [2]:
# Basic EDA
print("Dataset Shape:", df.shape)

print("\nIntent Distribution:")
print(df['intent'].value_counts())

print("\nMissing Values:")
print(df.isnull().sum())

df.head()

Dataset Shape: (160, 2)

Intent Distribution:
intent
Reschedule Flight    40
General Inquiry      40
Cancel Flight        40
Book Flight          40
Name: count, dtype: int64

Missing Values:
email     0
intent    0
dtype: int64


,email,intent
0,Reschedule my flight to next Monday.,Reschedule Flight
1,Reschedule my flight to next Monday.,Reschedule Flight
2,What is the baggage allowance?,General Inquiry
3,Please cancel my booking ABC689.,Cancel Flight
4,Reschedule my flight to next Monday.,Reschedule Flight


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Features and target
X = df['email']
y = df['intent']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# TF-IDF
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Train Logistic Regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Predictions
y_pred = model.predict(X_test_tfidf)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 1.0

Classification Report:
                   precision    recall  f1-score   support

      Book Flight       1.00      1.00      1.00         8
    Cancel Flight       1.00      1.00      1.00         8
  General Inquiry       1.00      1.00      1.00         8
Reschedule Flight       1.00      1.00      1.00         8

         accuracy                           1.00        32
        macro avg       1.00      1.00      1.00        32
     weighted avg       1.00      1.00      1.00        32



In [4]:
joblib.dump(model, "intent_model.pkl")
joblib.dump(tfidf, "tfidf.pkl")

print("Models saved successfully!")

Models saved successfully!


In [5]:
import os

print(os.listdir())

['.config', 'intent_model.pkl', 'tfidf.pkl', 'airline_emails.csv', 'sample_data']


In [6]:
import spacy
import re

nlp = spacy.load("en_core_web_sm")

def extract_entities(text):
    doc = nlp(text)

    entities = {
        "name": None,
        "source": None,
        "destination": None,
        "travel_date": None
    }

    # Passenger Name
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            entities["name"] = ent.text

        elif ent.label_ == "DATE":
            entities["travel_date"] = ent.text

    # Source and Destination
    match = re.search(r'from\s+(\w+)\s+to\s+(\w+)', text, re.IGNORECASE)

    if match:
        entities["source"] = match.group(1)
        entities["destination"] = match.group(2)

    return entities

In [7]:
sample = "Book a flight for John from Hyderabad to Delhi tomorrow."

print(extract_entities(sample))

{'name': 'John', 'source': 'Hyderabad', 'destination': 'Delhi', 'travel_date': 'tomorrow'}


In [8]:
import joblib

# Load saved models
model = joblib.load("intent_model.pkl")
tfidf = joblib.load("tfidf.pkl")

def analyze_email(text):
    # Intent Prediction
    text_vector = tfidf.transform([text])

    intent = model.predict(text_vector)[0]

    confidence = round(
        max(model.predict_proba(text_vector)[0]) * 100,
        2
    )

    # Entity Extraction
    entities = extract_entities(text)

    return {
        "intent": intent,
        "confidence": confidence,
        "entities": entities
    }

In [9]:
sample = "Book a flight for John from Hyderabad to Delhi tomorrow."

result = analyze_email(sample)

print(result)

{'intent': 'Book Flight', 'confidence': np.float64(86.11), 'entities': {'name': 'John', 'source': 'Hyderabad', 'destination': 'Delhi', 'travel_date': 'tomorrow'}}


In [10]:
import joblib

# Load saved models
model = joblib.load("intent_model.pkl")
tfidf = joblib.load("tfidf.pkl")

def analyze_email(text):
    # Intent Prediction
    text_vector = tfidf.transform([text])

    intent = model.predict(text_vector)[0]

    confidence = round(
        max(model.predict_proba(text_vector)[0]) * 100,
        2
    )

    # Entity Extraction
    entities = extract_entities(text)

    return {
        "intent": intent,
        "confidence": confidence,
        "entities": entities
    }

In [12]:
sample = "Book a flight for John from Hyderabad to Delhi tomorrow."

result = analyze_email(sample)

print(result)

{'intent': 'Book Flight', 'confidence': np.float64(86.11), 'entities': {'name': 'John', 'source': 'Hyderabad', 'destination': 'Delhi', 'travel_date': 'tomorrow'}}


In [13]:
from google.colab import files

files.download("intent_model.pkl")
files.download("tfidf.pkl")
files.download("airline_emails.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
%%writefile app.py

import streamlit as st
import joblib
import spacy
import re

# Load models
model = joblib.load("intent_model.pkl")
tfidf = joblib.load("tfidf.pkl")
nlp = spacy.load("en_core_web_sm")

def extract_entities(text):
    doc = nlp(text)

    entities = {
        "name": None,
        "source": None,
        "destination": None,
        "travel_date": None
    }

    for ent in doc.ents:
        if ent.label_ == "PERSON":
            entities["name"] = ent.text

        elif ent.label_ == "DATE":
            entities["travel_date"] = ent.text

    match = re.search(r'from\\s+(\\w+)\\s+to\\s+(\\w+)', text, re.IGNORECASE)

    if match:
        entities["source"] = match.group(1)
        entities["destination"] = match.group(2)

    return entities

st.title("✈️ Airline Ticket Booking Automation System")

email = st.text_area("Paste Customer Email")

if st.button("Analyze"):

    if email.strip():

        vector = tfidf.transform([email])

        intent = model.predict(vector)[0]

        confidence = round(
            max(model.predict_proba(vector)[0]) * 100,
            2
        )

        entities = extract_entities(email)

        st.success("Analysis Complete")

        st.write("### Intent")
        st.write(intent)

        st.write("### Confidence")
        st.write(f"{confidence}%")

        st.write("### Extracted Details")
        st.json(entities)

    else:
        st.warning("Please enter an email.")

Writing app.py


In [15]:
%%writefile requirements.txt

streamlit
scikit-learn
spacy
pandas
numpy
joblib
nltk

Writing requirements.txt


In [16]:
files.download("app.py")
files.download("requirements.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>